# Deepfake detection v3: prepare the data, fine-tune the model, measure before and after
**Before you press Run All:** Settings → Accelerator **GPU T4 x2** · Internet **On** · inputs added with **+ Add Input**:
- **A** manjilkarki/deepfake-and-real-images · **B** xhlulu/140k-real-and-fake-faces · **C** competition deepfake-detection-challenge
- **D** kaustubhdhote/human-faces-dataset (real and AI faces from newer generators)
- **E** ahnuf05/nano-banana-2-0-the-omni-subject-dataset (Gemini images; more are downloaded from Hugging Face in cell 3)
- **G** is downloaded automatically: a Gemini (Nano Banana 2) vs real benchmark, **test only**
- **H** (optional) your own **gemini-faces-test** dataset of Gemini faces you made, test only

About 2 hours in total. Every cell can be run again on its own after a restart: finished work is skipped.
At the end, download **model_v3.zip** and **results_v3.zip** (links appear under cell 7).

In [ ]:
# 1. Get our code and the one library Kaggle is missing (about 1 min)
import os
if os.path.isdir("/kaggle/working/code/.git"):
    !cd /kaggle/working/code && git pull -q
else:
    !git clone -q https://github.com/vidush-islockedin/DeepFake-detection /kaggle/working/code
!pip install -q facenet-pytorch==2.6.0 --no-deps
%cd /kaggle/working/code
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "NONE - turn on GPU T4 x2 in Settings")

In [ ]:
import os, sys
CODE = "/kaggle/working/code"
if not os.path.isdir(CODE + "/.git"):   # Kaggle restarted: get the code again, so this cell works on its own
    os.system(f"git clone -q https://github.com/vidush-islockedin/DeepFake-detection {CODE} && pip install -q facenet-pytorch==2.6.0 --no-deps")
os.chdir(CODE); sys.path.insert(0, CODE)
from kaggle_setup import *

# 2. Find the datasets wherever Kaggle mounted them (a few seconds; never opens the image folders)
P = datasets(refresh=True)

In [ ]:
import os, sys
CODE = "/kaggle/working/code"
if not os.path.isdir(CODE + "/.git"):   # Kaggle restarted: get the code again, so this cell works on its own
    os.system(f"git clone -q https://github.com/vidush-islockedin/DeepFake-detection {CODE} && pip install -q facenet-pytorch==2.6.0 --no-deps")
os.chdir(CODE); sys.path.insert(0, CODE)
from kaggle_setup import *

# 3. MTCNN face crops -> /kaggle/temp/faces/<train|val|test>/<A|B|C|D|E|G|H>/<real|fake>/   (about 25-35 min)
#    Two datasets at a time, one per GPU, while the Gemini images (E for training, G for testing) download. A progress line is shown every minute;
#    full logs are in /kaggle/temp/logs. Finished datasets and already-cropped images are skipped on a re-run.
P = datasets()
NB = "/kaggle/temp/nano_banana"
first, second = [], []
if not done("A"):
    first.append(("A", f'python prepare.py --src "{P["A"]}" --name A --out {FACES} --per-class 5000', True))
if not done("B"):
    first.append(("B", f'python prepare.py --src "{P["B"]}" --name B --out {FACES} --per-class 5000', True))
if not done("E"):
    first.append(("download_E", f"python fetch_hf.py --repo bitmind/nano-banana --out {NB} --max 4000", False))
GT = "/kaggle/temp/gemini_test"
if not done("G"):
    first.append(("download_G", f"python fetch_hf.py --repo zr-zhang/MLLM-Generated-Image-Detection-Dataset --meta "
                  f"--keep real=real nano_banana2=fake --out {GT} --max 1500", False))
if P.get("C") and not done("C"):
    second.append(("C", f'rm -rf {FACES}/*/C; python prepare.py --dfdc "{P["C"]}" --name C --out {FACES} --dfdc-frames 4', True))
if P.get("D") and not done("D"):
    second.append(("D", f'python prepare.py --src "{P["D"]}" --name D --out {FACES} --per-class 5000 --jpeg 0.5', True))
if P.get("H") and not done("H"):
    second.append(("H", f'python prepare.py --src "{P["H"]}" --name H --all-test --out {FACES}', True))
try:
    run(first)
except SystemExit as e:              # a failed Hugging Face download is not fatal: the other datasets still run
    print(f"{e}\nCarrying on. A failed download only means that dataset is left out (check Internet is On).")
E_SRC = " ".join(f'"{s}"' for s in (NB, P.get("E_KAGGLE")) if s and os.path.isdir(s) and os.listdir(s))
if E_SRC and not done("E"):
    second.append(("E", f"python prepare.py --src {E_SRC} --name E --label fake --out {FACES} --per-class 5000 --jpeg 0.5", True))
if os.path.isdir(GT) and os.listdir(GT) and not done("G"):     # test only: never used for training
    second.append(("G", f'python prepare.py --src "{GT}" --name G --all-test --per-class 1500 --out {FACES}', True))
run(second)
assert all(done(n) for n in "AB"), "A or B did not finish - run this cell again"
_ = counts()

In [ ]:
import os, sys
CODE = "/kaggle/working/code"
if not os.path.isdir(CODE + "/.git"):   # Kaggle restarted: get the code again, so this cell works on its own
    os.system(f"git clone -q https://github.com/vidush-islockedin/DeepFake-detection {CODE} && pip install -q facenet-pytorch==2.6.0 --no-deps")
os.chdir(CODE); sys.path.insert(0, CODE)
from kaggle_setup import *

# 4. BEFORE: the downloaded pre-trained model, untouched (about 5 min)
!python evaluate.py --model dima806/deepfake_vs_real_image_detection --data {FACES}/test --out /kaggle/working/results/before

In [ ]:
import os, sys
CODE = "/kaggle/working/code"
if not os.path.isdir(CODE + "/.git"):   # Kaggle restarted: get the code again, so this cell works on its own
    os.system(f"git clone -q https://github.com/vidush-islockedin/DeepFake-detection {CODE} && pip install -q facenet-pytorch==2.6.0 --no-deps")
os.chdir(CODE); sys.path.insert(0, CODE)
from kaggle_setup import *

# 5. Fine-tune on A + B + D + E + the training part of C (about 60-80 min on two T4s)
#    Warm-up + cosine learning rate, gradient clipping, best epoch by balanced accuracy, both GPUs.
#    If Kaggle stops here, run this cell again: it carries on from the last finished epoch.
!python train.py --data {FACES} --out /kaggle/working/model --epochs 3 --bs 32 --resume /kaggle/temp/ckpt.pt

In [ ]:
import os, sys
CODE = "/kaggle/working/code"
if not os.path.isdir(CODE + "/.git"):   # Kaggle restarted: get the code again, so this cell works on its own
    os.system(f"git clone -q https://github.com/vidush-islockedin/DeepFake-detection {CODE} && pip install -q facenet-pytorch==2.6.0 --no-deps")
os.chdir(CODE); sys.path.insert(0, CODE)
from kaggle_setup import *

# 6. AFTER: our fine-tuned model on the same test sets (about 5 min)
!python evaluate.py --model /kaggle/working/model --data {FACES}/test --out /kaggle/working/results/after
import json
print(f"{'set':4s} {'faces':>6s} {'before':>7s} {'after':>7s} {'AUC':>6s} {'FP':>4s} {'FN':>4s}")
b, a = (json.load(open(f"/kaggle/working/results/{n}/results.json")) for n in ("before", "after"))
for k, v in a.items():
    if isinstance(v, dict):
        auc = f"{v['auc']:.3f}" if v["auc"] is not None else "  n/a"
        print(f"{k:4s} {v['images']:6d} {b[k]['accuracy']:7.1%} {v['accuracy']:7.1%} {auc:>6s} {v['real_called_fake']:4d} {v['fake_called_real']:4d}")
print("cut-off", a["threshold"])

In [ ]:
import os, sys
CODE = "/kaggle/working/code"
if not os.path.isdir(CODE + "/.git"):   # Kaggle restarted: get the code again, so this cell works on its own
    os.system(f"git clone -q https://github.com/vidush-islockedin/DeepFake-detection {CODE} && pip install -q facenet-pytorch==2.6.0 --no-deps")
os.chdir(CODE); sys.path.insert(0, CODE)
from kaggle_setup import *

# 7. Pack the two files to download (model about 170 MB: 16-bit weights)
%cd /kaggle/working
!echo v3 > model/version.txt
!cp model/history.csv model/threshold.json results/ && rm -f model_v3.zip results_v3.zip && zip -qr model_v3.zip model && zip -qr results_v3.zip results && ls -lh *.zip
from IPython.display import FileLink, display
display(FileLink("model_v3.zip"), FileLink("results_v3.zip"))   # click these if the Output panel download fails